# Assignment 04 Solutions — Linear Systems: Gauss and Classification

Aligned with `assignments/ASSIGNMENTS.tex`. Replace TODOs with complete solutions.


## Tasks
- Implement escalonar(Ab) and substituicao_retroativa(U,b); solve system.
- Construct SPI system and show free variable solutions.
- Construct SI system and explain contradiction.
- Implement classificar_sistema(A,b) using rank analysis; test 5 systems.
- Give a parametric solution for a 3-unknowns SPI system.
- Compare with built-in solver; show singular behavior.
- Create 2D and 3D visualizations of systems.
- Write reflexao.md (200–300 words).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../packages/python/src").resolve()))

import numpy as np
from linalg_utils.systems import escalonar, substituicao_retroativa, resolver_gauss

# Implement escalonar and substituicao_retroativa; solve system
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

# Build augmented matrix and echelon
Ab = np.column_stack([A, b])
U_aug, steps = escalonar(Ab)
print("Echelon form:\n", U_aug)
for s in steps:
    print(f"  {s}")

# Back-substitution
U = U_aug[:, :-1]
c = U_aug[:, -1]
x = substituicao_retroativa(U, c)
print(f"\nSolution: x = {x}")
print(f"Verification A @ x = {A @ x}  (should be {b})")
np.testing.assert_allclose(A @ x, b, atol=1e-9)

In [ ]:
from linalg_utils.systems import classificar_sistema

# SPI system — infinitely many solutions
A_spi = np.array([[1, 2, 3], [2, 4, 6]], dtype=float)
b_spi = np.array([6, 12], dtype=float)
print(f"SPI system classification: {classificar_sistema(A_spi, b_spi)}")

# Free variable: x3 = t, x2 = s, x1 = 6 - 2s - 3t
print("Parametric solution: x1 = 6 - 2s - 3t, x2 = s, x3 = t")
for t in [0, 1, 2]:
    s = 0
    x = np.array([6 - 2*s - 3*t, s, t])
    print(f"  t={t}: x={x}, A@x={A_spi @ x}")

In [ ]:
# SI system — inconsistent, no solution
A_si = np.array([[1, 2], [2, 4]], dtype=float)
b_si = np.array([3, 7], dtype=float)
print(f"SI system classification: {classificar_sistema(A_si, b_si)}")

# Explanation: row 2 is 2*row 1 for A, but 7 != 2*3 = 6
# After elimination: 0*x1 + 0*x2 = 1, which is a contradiction.
Ab_si = np.column_stack([A_si, b_si])
U_si, _ = escalonar(Ab_si)
print(f"Echelon form:\n{U_si}")
print("Last row gives 0 = 1, a contradiction. System is inconsistent.")

In [ ]:
# classificar_sistema on 5 different systems
systems = [
    ("SPD 2x2", np.array([[2, 1], [1, 3]], dtype=float), np.array([5, 7], dtype=float)),
    ("SPD 3x3", np.array([[1, 0, 0], [0, 2, 0], [0, 0, 3]], dtype=float), np.array([1, 4, 9], dtype=float)),
    ("SPI", np.array([[1, 2], [2, 4]], dtype=float), np.array([3, 6], dtype=float)),
    ("SI", np.array([[1, 2], [2, 4]], dtype=float), np.array([3, 7], dtype=float)),
    ("SPD 3x3 dense", np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float), np.array([8, -11, -3], dtype=float)),
]

for name, A, b in systems:
    cls = classificar_sistema(A, b)
    print(f"{name}: {cls}")

In [ ]:
# Parametric solution for a 3-unknowns SPI system
# x + y + z = 3 and 2x + 2y + 2z = 6
A_param = np.array([[1, 1, 1], [2, 2, 2]], dtype=float)
b_param = np.array([3, 6], dtype=float)
print(f"Classification: {classificar_sistema(A_param, b_param)}")

# Parametric form: x = 3 - s - t, y = s, z = t
print("Solution: x = 3 - s - t, y = s, z = t")
for s, t in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    x = np.array([3 - s - t, s, t])
    print(f"  s={s}, t={t}: x={x}, A@x={A_param @ x}")

In [ ]:
# Compare with built-in solver; show singular behavior
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

x_gauss, cls, _ = resolver_gauss(A, b)
x_numpy = np.linalg.solve(A, b)

print(f"Gauss: x = {x_gauss}")
print(f"NumPy: x = {x_numpy}")
np.testing.assert_allclose(x_gauss, x_numpy, atol=1e-9)

# Singular system
A_sing = np.array([[1, 2], [2, 4]], dtype=float)
b_sing = np.array([3, 7], dtype=float)
x_g, cls_g, _ = resolver_gauss(A_sing, b_sing)
print(f"\nSingular system: classification={cls_g}, solution={x_g}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 2D visualization: two lines intersecting
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 2D: 2x + y = 5, x + 3y = 7
x_range = np.linspace(-1, 5, 100)
axes[0].plot(x_range, 5 - 2*x_range, label="2x + y = 5")
axes[0].plot(x_range, (7 - x_range)/3, label="x + 3y = 7")
axes[0].plot(1.6, 1.8, "ro", markersize=8, label="solution (1.6, 1.8)")
axes[0].set_title("2D System (SPD)")
axes[0].legend()
axes[0].grid(True)

# 3D visualization
ax3d = fig.add_subplot(122, projection="3d")
xx, yy = np.meshgrid(np.linspace(-2, 4, 20), np.linspace(-2, 6, 20))
# 2x + y - z = 8
zz1 = 2*xx + yy - 8
# -3x - y + 2z = -11
zz2 = (-11 + 3*xx + yy) / 2
ax3d.plot_surface(xx, yy, zz1, alpha=0.3, color="blue")
ax3d.plot_surface(xx, yy, zz2, alpha=0.3, color="red")
ax3d.scatter([2], [3], [-1], color="black", s=50, zorder=5)
ax3d.set_title("3D System (SPD)")
ax3d.set_xlabel("x"); ax3d.set_ylabel("y"); ax3d.set_zlabel("z")

plt.tight_layout()
plt.savefig("sistemas_visualizacao.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved sistemas_visualizacao.png")

In [ ]:
# Reflexao — see reflexao.md
# Summary: Gaussian elimination systematically reduces a system to echelon form,
# making the classification (SPD, SPI, SI) clear from the pivot structure.
# Rank analysis via the echelon form determines whether solutions exist and
# whether they are unique. Comparing with NumPy's solver confirmed numerical
# accuracy and highlighted that singular systems require special handling.